In [ ]:
from pathlib import Path
import random
from ultralytics.models import YOLO
import torch
import yaml

random.seed(42)

In [ ]:
from utils import DisplayPath
Path = DisplayPath

## Paths and hyperparameters

In [ ]:
# Created by e2e_data_prep.ipynb
YOLO_DATASET = Path("datasets/ready/full_dataset")
RUNS_DIR = Path("runs/segment")

if not YOLO_DATASET.exists():
    raise FileNotFoundError(f"Dataset not found at {YOLO_DATASET}, run e2e_data_prep.ipynb first")

YOLO_DATASET.display()

In [ ]:
BATCH_SIZE = 16
IMG_SIZE = 640
model_type = "yolo11n-seg.pt"
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")

## Dataset check

In [ ]:
stats = {}

for split in ['train', 'val', 'test']:
    img_dir = YOLO_DATASET / split / "images"
    lbl_dir = YOLO_DATASET / split / "labels"

    if img_dir.exists() and lbl_dir.exists():
        stats[split] = {'images': len(list(img_dir.glob("*"))), 'labels': len(list(lbl_dir.glob("*.txt")))}
        print(f"{split:5s}: {stats[split]['images']:4d} images, {stats[split]['labels']:4d} labels")
    else:
        stats[split] = {'images': 0, 'labels': 0}
        print(f"{split:5s}: missing")

total_images = sum(s['images'] for s in stats.values())
if total_images == 0:
    raise RuntimeError("Empty dataset, run e2e_data_prep.ipynb first")

## Class distribution

In [ ]:
from src.data_utils import count_class_instances

class_names = {0: 'Red Ball', 1: 'Human', 2: 'Trashcan'}

counts = count_class_instances(YOLO_DATASET, 'train')
total = sum(counts.values())

for class_id, count in counts.items():
    percentage = (count / total * 100) if total > 0 else 0
    print(f"{class_names[class_id]:12s} (class {class_id}): {count:5d} instances ({percentage:5.1f}%)")
print(f"{'Total':12s}           : {total:5d} instances")

if counts[2] > 0:
    print(f"Imbalance ratio: {max(counts.values()) / min(v for v in counts.values() if v > 0):.1f}x")

In [ ]:
import albumentations as A
from src.augmentation import augment_class_dataset, augment_class_with_occlusion

trashcan_pipeline = A.Compose([  # type: ignore
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.8),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
    A.RandomScale(scale_limit=0.3, p=0.7),
    A.Affine(rotate=(-20, 20), translate_percent=0.1, scale=(0.8, 1.2), shear=(-10, 10), p=0.7),
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

DO_AUGMENTATION = True
if DO_AUGMENTATION:
    augment_class_dataset(
        dataset_path=YOLO_DATASET,
        class_id=2,
        num_augmentations=20,
        aug_config=trashcan_pipeline
    )

    # Smaller and partially hidden balls, labels are cut accordingly
    augment_class_with_occlusion(
        dataset_path=YOLO_DATASET,
        class_id=0,
        num_augmentations=5
    )

### Remove augmented images

Run this before augmenting again, otherwise copies pile up.

In [ ]:
from src.augmentation import clean_augmented_images

CLEAN_AUGMENTED = False

if CLEAN_AUGMENTED:
    removed = clean_augmented_images(YOLO_DATASET)

## data.yaml

In [ ]:
classes = {
    'red ball': 0,
    'human': 1,
    'trashcan': 2
}

config = {
    'path': str(YOLO_DATASET.absolute()),
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(classes),
    'names': list(classes.keys())
}

config_path = YOLO_DATASET / 'data.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

config_path.display()

## Monitoring images

A few validation images per class, predicted at the end of each epoch.

In [ ]:
from src.data_utils import select_diverse_monitoring_images

MONITOR_IMAGES = select_diverse_monitoring_images(
    YOLO_DATASET / 'val' / 'labels',
    YOLO_DATASET / 'val' / 'images',
    images_per_class=3,
    include_mixed=True
)

In [ ]:
from src.visualization import create_monitoring_callback

## Training

Backbone frozen, only the head is trained.

In [ ]:
model = YOLO(model_type)

In [ ]:
project_name = 'ball_person_trashcan_model_v5'

In [ ]:
monitor_output_dir = RUNS_DIR / project_name / 'training_monitor'

callback_fn = create_monitoring_callback(
    model=model,
    monitor_images=MONITOR_IMAGES,
    output_dir=monitor_output_dir,
    project_name=project_name
)
model.add_callback('on_train_epoch_end', callback_fn)

In [ ]:
head_idx = next((i for i, m in enumerate(model.model.model) if 'Detect' in m.__class__.__name__ or 'Segment' in m.__class__.__name__), len(model.model.model) - 1)

results = model.train(
    data=str(config_path),
    epochs=10,
    freeze=list(range(head_idx)),
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=project_name,
    exist_ok=True,
    save=True,
    save_period=5,
    val=True,
    optimizer='Adam',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    box=7.5,
    cls=1.0,  # was 20.0 before trashcans were augmented
    dfl=1.5,
    patience=20,
    workers=8,
    verbose=True
)

## Evaluation

In [ ]:
best_model_path = RUNS_DIR / project_name / 'weights' / 'best.pt'
best_model_path.display()
model = YOLO(best_model_path)

In [ ]:
metrics = model.val()

print(f"Box mAP50: {metrics.box.map50:.4f}")
print(f"Box mAP50-95: {metrics.box.map:.4f}")
print(f"Mask mAP50: {metrics.seg.map50:.4f}")
print(f"Mask mAP50-95: {metrics.seg.map:.4f}")

for i, class_name in enumerate(['red ball', 'human', 'trashcan']):
    try:
        map50 = metrics.seg.map50_per_class[i] if hasattr(metrics.seg, 'map50_per_class') else 0
        map_val = metrics.seg.map_per_class[i] if hasattr(metrics.seg, 'map_per_class') else 0
        print(f"{class_name:12s}: mAP50={map50:.4f}, mAP50-95={map_val:.4f}")
    except IndexError:
        print(f"{class_name:12s}: no metrics")

In [ ]:
model_dir = RUNS_DIR / project_name
(model_dir / 'weights' / 'best.pt').display()
(model_dir / 'weights' / 'last.pt').display()
(model_dir / 'training_monitor').display()

## Monitoring images per epoch

In [ ]:
monitor_dir = RUNS_DIR / project_name / 'training_monitor'

for viz_file in sorted(monitor_dir.glob("epoch_*.jpg")):
    viz_file.display()

## Predictions on the validation set

In [ ]:
for img_path in (YOLO_DATASET / "val" / "images").glob("*"):
    model.predict(str(img_path), save=True, conf=0.1)

(RUNS_DIR / project_name).display()